# 🌱 Greenhouse IoT — Phân tích chuyên sâu 3 mô hình Machine Learning

Notebook này phân tích toàn diện hiệu suất của 3 mô hình dự đoán nhu cầu tưới nước:
1. **Linear Regression**
2. **Random Forest**
3. **XGBoost**

---
### Hướng dẫn:
1. Nhấn **Runtime → Run all** (hoặc Ctrl+F9).
2. Khi cell "Upload dữ liệu" chạy, hãy chọn file **DATA.csv** từ máy tính.
3. Toàn bộ 15 phần phân tích sẽ tự động hiển thị.

## 1. Cài đặt & Import thư viện

In [ ]:
!pip install -q xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
import warnings
import os
warnings.filterwarnings('ignore')

# Cấu hình chung
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

COLORS = {'linear': '#3b82f6', 'random_forest': '#22c55e', 'xgboost': '#f97316'}
MODEL_NAMES = {'linear': 'Linear Regression', 'random_forest': 'Random Forest', 'xgboost': 'XGBoost'}
FEATURE_COLUMNS = ['Humidity', 'Atmospheric_Temp', 'Soil_Temp', 'Soil_Moisture', 'Dew_Point']
TARGET_COLUMN = 'Water_Need'
RANDOM_STATE = 42
print('✅ Đã chuẩn bị thư viện.')

## 2. Upload & Load dữ liệu

In [ ]:
if not os.path.exists('DATA.csv'):
    from google.colab import files
    print('📁 Hãy chọn file DATA.csv...')
    uploaded = files.upload()
    if 'DATA.csv' not in uploaded:
        os.rename(list(uploaded.keys())[0], 'DATA.csv')

df = pd.read_csv('DATA.csv')
print(f'📊 Load thành công {len(df):,} bản ghi.')
df.head()

## 3. EDA - Phân phối các biến (Histograms)

In [ ]:
all_cols = FEATURE_COLUMNS + [TARGET_COLUMN]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for idx, col in enumerate(all_cols):
    ax = axes[idx // 3][idx % 3]
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color=list(COLORS.values())[idx % 3])
    ax.set_title(f'Phân phối: {col}')
plt.tight_layout()
plt.show()

## 4. Ma trận tương quan

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[all_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('🔗 Ma trận tương quan giữa các biến', fontweight='bold')
plt.show()

## 5. Quan hệ đặc trưng vs Target

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for idx, col in enumerate(FEATURE_COLUMNS):
    sns.regplot(data=df.sample(min(1000, len(df))), x=col, y=TARGET_COLUMN, ax=axes[idx], 
                scatter_kws={'alpha':0.2, 's':10}, line_kws={'color':'red'})
plt.tight_layout()
plt.show()

## 6. Huấn luyện & Đánh giá (R², RMSE, MAE)

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)

models = {
    'linear': LinearRegression(),
    'random_forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE),
    'xgboost': XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE)
}

results = {}
predictions = {}

for key, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[key] = y_pred
    results[key] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R²': r2_score(y_test, y_pred)
    }
    print(f'Done: {MODEL_NAMES[key]}')

res_df = pd.DataFrame(results).T
res_df

## 7. Biểu đồ cột so sánh

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, m in enumerate(['R²', 'RMSE', 'MAE']):
    sns.barplot(x=res_df.index, y=res_df[m], ax=axes[idx], palette=list(COLORS.values()))
    axes[idx].set_title(m, fontweight='bold')
plt.show()

## 8. Tầm quan trọng của các đặc trưng

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (key, model) in enumerate(models.items()):
    imp = model.feature_importances_ if hasattr(model, 'feature_importances_') else np.abs(model.coef_)
    sns.barplot(x=imp, y=FEATURE_COLUMNS, ax=axes[idx], color=COLORS[key])
    axes[idx].set_title(f'Tầm quan trọng: {MODEL_NAMES[key]}')
plt.tight_layout()
plt.show()

## 9. Cross-Validation (5-Fold)

In [ ]:
cv_scores = {}
for key, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    cv_scores[key] = scores

plt.figure(figsize=(10, 5))
plt.boxplot([cv_scores[k] for k in cv_scores.keys()], labels=[MODEL_NAMES[k] for k in cv_scores.keys()])
plt.title('📦 Cross-Validation R² (5-Fold)', fontweight='bold')
plt.show()

## 10. Learning Curve (Overfitting Analysis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for idx, (key, model) in enumerate(models.items()):
    sz, train_sc, val_sc = learning_curve(model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 5))
    axes[idx].plot(sz, train_sc.mean(axis=1), 'o-', label='Train')
    axes[idx].plot(sz, val_sc.mean(axis=1), 'o-', label='Val')
    axes[idx].set_title(f'Learning Curve: {MODEL_NAMES[key]}')
    axes[idx].legend()
plt.show()

## 11. Residual Plot (Phân tích sai số)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, key in enumerate(models.keys()):
    sns.scatterplot(x=predictions[key], y=y_test - predictions[key], ax=axes[idx], color=COLORS[key], alpha=0.3)
    axes[idx].axhline(0, color='red', ls='--')
    axes[idx].set_title(f'Residuals: {MODEL_NAMES[key]}')
plt.show()

## 12. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, key in enumerate(models.keys()):
    axes[idx].scatter(y_test, predictions[key], alpha=0.3, color=COLORS[key])
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
    axes[idx].set_title(f'Actual vs Pred: {MODEL_NAMES[key]}')
plt.show()

## 13. MA TRẬN NHẦM LẪN (Dựa trên logic nhị phân)
Quy đổi từ con số dự báo sang quyết định **Tưới (1)** hoặc **Không tưới (0)** bằng ngưỡng Threshold.

In [ ]:
THRESHOLD = 0.5
y_test_bin = (y_test >= THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, key in enumerate(models.keys()):
    y_pred_bin = (predictions[key] >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test_bin, y_pred_bin)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    acc = (cm[0,0]+cm[1,1])/cm.sum()
    axes[idx].set_title(f'{MODEL_NAMES[key]}\nAccuracy Logic: {acc:.2%}')
    axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 14. Biểu đồ đường so sánh (Overlay)

In [ ]:
plt.figure(figsize=(16, 6))
n = 150
plt.plot(y_test.values[:n], 'k--', label='Actual', alpha=0.6)
for k in models.keys():
    plt.plot(predictions[k][:n], label=MODEL_NAMES[k], color=COLORS[k])
plt.title(f'So sánh 150 mẫu đầu tiên', fontweight='bold')
plt.legend()
plt.show()

## 15. Kết luận
Dựa trên R² và RMSE, ta có thể kết luận mô hình phù hợp nhất cho hệ thống.